In [1]:
import pickle
from pathlib import Path

import jax.numpy as jnp
import numpy as np
from jax import config

config.update("jax_enable_x64", True)


import polars as pl

# from scipy.stats import norm


In [2]:
RESULTS_DIR = Path("..") / "data" / "results" / "toy_example"
PLOT_DIR = Path("..") / "figures"

In [ ]:
true_flux = np.load(RESULTS_DIR / "true_flux.npy")
true_mid = np.load(RESULTS_DIR / "true_composition.npy")
print(f"True flux: {true_flux}")
print(f"True isotopologue distribution: {true_mid}")


True flux: [100. 110.  50.  20.  20.  80.]
True isotopologue distribution: [6.34920635e-05 8.00761905e-01 1.98285714e-01 8.88888889e-04]


We start with analysis of the reference flux set from Antoniwitch 

In [ ]:
intensity_msts = jnp.load(RESULTS_DIR / "intensity_measurements_reference.npy")
simulated_mid = jnp.load(RESULTS_DIR / "mid_simulated_reference.npy")

In [8]:
with open(RESULTS_DIR / "estimated_fluxes_reference.pkl", "rb") as f:
    estimated_fluxes_ref = pickle.load(f)
estimated_fluxes_ref["trad"].shape

(10000, 6)

In [11]:
# Column order in the estimated_fluxes_ref array
col_order = ["v2", "v4", "v6", "v1", "v3", "v5"]
col_index = {name: i for i, name in enumerate(col_order)}

# true_flux array is ordered v1..v6
true_flux_map = {f"v{i+1}": float(true_flux[i]) for i in range(6)}

flux_keys = ["v2", "v3", "v4", "v5", "v6"]  # v1 excluded

rows = []
for model in ("trad", "ilr"):
    arr = np.array(estimated_fluxes_ref[model])  # shape (N, 6)
    for flux_name in flux_keys:
        col = arr[:, col_index[flux_name]]
        rows.append({
            "model": model,
            "flux": flux_name,
            "true_value": true_flux_map[flux_name],
            "median": float(np.median(col)),
            "min": float(np.min(col)),
            "max": float(np.max(col)),
            "q25": float(np.quantile(col, 0.25)),
            "q75": float(np.quantile(col, 0.75)),
        })

summary_df_toy = pl.DataFrame(rows)
summary_df_toy.write_csv(RESULTS_DIR / "estimated_fluxes_reference_summary.csv")
print(summary_df_toy.to_pandas().to_string(index=False))

model flux  true_value     median       min        max        q25        q75
 trad   v2       110.0 109.844672 72.563415 265.517116 100.731467 120.875892
 trad   v3        50.0  49.986073 24.443107 193.630232  42.847721  58.922381
 trad   v4        20.0  20.015206 14.012285  26.861354  18.888643  21.167835
 trad   v5        20.0  20.015206 14.012285  26.861354  18.888643  21.167835
 trad   v6        80.0  79.984794 73.138646  85.987715  78.832165  81.111357
  ilr   v2       110.0 110.068021 73.398209 267.962011 101.183480 120.054613
  ilr   v3        50.0  50.018433 25.076801 196.058404  43.518100  57.895028
  ilr   v4        20.0  20.013511 14.048196  26.758693  18.894809  21.153746
  ilr   v5        20.0  20.013511 14.048196  26.758693  18.894809  21.153746
  ilr   v6        80.0  79.986489 73.241307  85.951804  78.846254  81.105191


In [12]:
col_rename = {
    "flux": "Flux",
    "true_value": "True value",
    "median": "Median",
    "min": "Min",
    "max": "Max",
    "q25": "Q25",
    "q75": "Q75",
}

for model, label in (("trad", "Traditional I-MFA"), ("ilr", "Compositional I-MFA")):
    df = (
        summary_df_toy.filter(pl.col("model") == model)
        .drop("model")
        .rename(col_rename)
        .to_pandas()
    )
    latex = df.to_latex(
        index=False,
        float_format="%.3f",
        caption=f"Estimated fluxes — {label}",
        label=f"tab:toy_fluxes_{model}",
    )
    print(f"% --- {label} ---")
    print(latex)

% --- Traditional I-MFA ---
\begin{table}
\caption{Estimated fluxes — Traditional I-MFA}
\label{tab:toy_fluxes_trad}
\begin{tabular}{lrrrrrr}
\toprule
Flux & True value & Median & Min & Max & Q25 & Q75 \\
\midrule
v2 & 110.000 & 109.845 & 72.563 & 265.517 & 100.731 & 120.876 \\
v3 & 50.000 & 49.986 & 24.443 & 193.630 & 42.848 & 58.922 \\
v4 & 20.000 & 20.015 & 14.012 & 26.861 & 18.889 & 21.168 \\
v5 & 20.000 & 20.015 & 14.012 & 26.861 & 18.889 & 21.168 \\
v6 & 80.000 & 79.985 & 73.139 & 85.988 & 78.832 & 81.111 \\
\bottomrule
\end{tabular}
\end{table}

% --- Compositional I-MFA ---
\begin{table}
\caption{Estimated fluxes — Compositional I-MFA}
\label{tab:toy_fluxes_ilr}
\begin{tabular}{lrrrrrr}
\toprule
Flux & True value & Median & Min & Max & Q25 & Q75 \\
\midrule
v2 & 110.000 & 110.068 & 73.398 & 267.962 & 101.183 & 120.055 \\
v3 & 50.000 & 50.018 & 25.077 & 196.058 & 43.518 & 57.895 \\
v4 & 20.000 & 20.014 & 14.048 & 26.759 & 18.895 & 21.154 \\
v5 & 20.000 & 20.014 & 14.048 & 26.759